# CAMPFIRE Python Client — Quick Start

This notebook demonstrates the CAMPFIRE Python client for querying, downloading, and analyzing NIRSpec spectroscopic data.

**Prerequisites:** Install and authenticate before running this notebook:
```bash
pip install "git+https://github.com/hollisakins/campfire.git#subdirectory=python/"
campfire login
```

In [ ]:
from campfire import Campfire

cf = Campfire()
print(f"Local data available: {cf.is_local}")
print(f"Last synced: {cf.last_synced}")

## 1. Sync the Catalog

Pull the full object/spectra catalog into a local SQLite database. This is metadata only — no FITS files — and takes a few seconds. Safe to run often.

In [ ]:
result = cf.sync()
print(f"Synced {result['observations']} observations, "
      f"{result['objects']} objects, "
      f"{result['spectra']} spectra")

if result['stale_count'] > 0:
    print(f"\n⚠ {result['stale_count']} local files have been reprocessed on the server.")
    print("  Run: cf.download(stale_only=True)")

## 2. Query Objects

After syncing, all queries run against your local SQLite database — instant, no network needed. Results are returned as an `astropy.table.Table`.

In [ ]:
# High-redshift galaxies with good redshift measurements
high_z = cf.query_objects(
    redshift_range=(3.0, 10.0),
    redshift_quality=['probable', 'secure'],  # or [3, 4]
    inspected_only=True,
)

print(f"Found {len(high_z)} high-z galaxies")
high_z['object_id', 'field', 'redshift', 'redshift_quality', 'max_snr'][:10]

In [ ]:
# Filter by program and field (case-insensitive)
cosmos = cf.query_objects(
    fields=['COSMOS'],
    redshift_range=(4.0, 8.0),
)

print(f"Found {len(cosmos)} COSMOS objects at z=4-8")

## 3. Flag Filtering

CAMPFIRE uses bitmask flags for spectral features, object classifications, and data quality. The Python client provides numpy-style operators for intuitive filtering.

In [ ]:
from campfire import list_flags

# See all available flags
list_flags()

In [ ]:
from campfire import ObjectFlags, SpectralFeatures, DQFlags

# Find LRDs or Lyman-alpha emitters, excluding broad-line AGN
interesting = cf.query_objects(
    object_flags=(ObjectFlags.LRD | ObjectFlags.LYA_EMITTER) & ~ObjectFlags.BROAD_LINE,
)
print(f"Found {len(interesting)} LRDs/LAEs (no broad lines)")

# Objects with multiple emission lines and clean data
clean_emitters = cf.query_objects(
    spectral_features=SpectralFeatures.MULTI_EMISSION,
    dq_flags=~(DQFlags.CONTAMINATION | DQFlags.LOW_SNR),
)
print(f"Found {len(clean_emitters)} clean multi-emission objects")

In [ ]:
from campfire import decode_flags

# String-based filtering (simpler, same result)
lrds = cf.query_objects(object_flags=['LRD'])
print(f"Found {len(lrds)} LRDs")

# Decode a bitmask value from results
if len(lrds) > 0:
    flags_val = lrds[0]['object_flags']
    print(f"First object flags: {decode_flags(flags_val, 'object_flags')}")

## 4. Cone Search

In [ ]:
# Find objects within 30 arcsec of a coordinate
nearby = cf.query_objects(
    cone_search=(150.0832, 2.3511, 30.0),  # RA, Dec (deg), radius (arcsec)
)

print(f"Found {len(nearby)} objects within 30\"")
if len(nearby) > 0:
    nearby['object_id', 'ra', 'dec', 'redshift'][:5]

## 5. Bulk Download FITS Files

Use `cf.download()` to fetch FITS files in bulk. This uses the manifest API (one request per observation) and downloads in parallel — much faster than downloading one at a time.

In [ ]:
# Download all PRISM spectra for a specific observation
cf.download(
    observations=['ember_cosmos_p1'],
    gratings=['PRISM'],
)

In [ ]:
# Other download options:
# cf.download(programs=['EMBER-UDS'])              # By program
# cf.download(fields=['COSMOS'])                    # By field
# cf.download(fields=['COSMOS'], gratings=['PRISM', 'G395M'])  # Multiple gratings
# cf.download(stale_only=True)                      # Re-download reprocessed files

## 6. Open a Spectrum

`open_spectrum()` is the single entry point for accessing spectral data. It checks for locally downloaded files first; if not found, it downloads from the API and caches locally so subsequent calls are instant.

In [ ]:
import matplotlib.pyplot as plt

# Open a spectrum — uses local file if downloaded, otherwise fetches from API
spec = cf.open_spectrum('ember_cosmos_p1_920424', 'PRISM')
print(spec)

# Access arrays
print(f"Wavelength: {spec.wavelength.shape}")
print(f"Flux (fnu): {spec.flux.shape}")
print(f"Flux err:   {spec.flux_err.shape}")
print(f"Flam:       {'available' if spec.flam is not None else 'not available'}")

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(12, 5))

# Mask bad pixels for cleaner plotting
good = np.isfinite(spec.flux) & np.isfinite(spec.flux_err) & (spec.flux_err > 0)

ax.plot(spec.wavelength[good], spec.flux[good], lw=0.8, color='k')
ax.fill_between(
    spec.wavelength[good],
    spec.flux[good] - spec.flux_err[good],
    spec.flux[good] + spec.flux_err[good],
    alpha=0.3,
)
ax.set_xlabel('Wavelength (μm)')
ax.set_ylabel('f$_\\nu$ (μJy)')
ax.set_title(f'{spec.object_id} — {spec.grating}')
ax.axhline(0, color='gray', lw=0.5, ls='--')
plt.tight_layout()
plt.show()

In [ ]:
# Second call is instant — file is cached locally
spec2 = cf.open_spectrum('ember_cosmos_p1_920424', 'PRISM')
print(f"Same data: {np.array_equal(spec.wavelength, spec2.wavelength)}")

## 7. Open Any FITS File Directly

`SpectrumData.from_fits()` works with any CAMPFIRE pipeline FITS file — useful if you have files from the pipeline or collaborators.

In [ ]:
from campfire import SpectrumData

# Open any FITS file directly (no client needed)
# spec = SpectrumData.from_fits('/path/to/my_spectrum.fits')
# print(spec)

## 8. Redshift Distribution

In [ ]:
# Query all inspected objects
inspected = cf.query_objects(
    redshift_quality=['tentative', 'probable', 'secure'],  # or [2, 3, 4]
    inspected_only=True,
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(inspected['redshift'], bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Redshift')
ax.set_ylabel('Count')
ax.set_title(f'Redshift Distribution ({len(inspected)} inspected objects)')
plt.tight_layout()
plt.show()

## 9. Iterate Over All Matching Objects

`iter_objects()` streams through all matching objects without loading everything into memory. Useful for large result sets.

In [ ]:
# Count H-alpha emitters across the full catalog
count = 0
for obj in cf.iter_objects(object_flags=ObjectFlags.HA_EMITTER):
    count += 1

print(f"Total H-alpha emitters: {count}")

## 10. Cross-Match with External Catalog

In [ ]:
from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u

# Get CAMPFIRE objects in COSMOS (case-insensitive)
campfire_objects = cf.query_objects(fields=['COSMOS'])

# Your catalog (example)
my_catalog = Table({
    'id': ['src1', 'src2', 'src3'],
    'ra': [150.08, 150.12, 150.15],
    'dec': [2.35, 2.40, 2.45],
})

# Cross-match within 1 arcsec
my_coords = SkyCoord(ra=my_catalog['ra'] * u.deg, dec=my_catalog['dec'] * u.deg)
cf_coords = SkyCoord(ra=campfire_objects['ra'] * u.deg, dec=campfire_objects['dec'] * u.deg)

idx, sep, _ = my_coords.match_to_catalog_sky(cf_coords)
matched = sep < 1 * u.arcsec

print(f"Matched {matched.sum()} / {len(my_catalog)} sources within 1\"")

if matched.sum() > 0:
    matches = Table()
    matches['my_id'] = my_catalog['id'][matched]
    matches['campfire_id'] = campfire_objects['object_id'][idx[matched]]
    matches['separation_arcsec'] = sep[matched].to(u.arcsec)
    matches['redshift'] = campfire_objects['redshift'][idx[matched]]
    print(matches)

## 11. CSV Workflow

After `campfire sync` (or `cf.sync()`), CSV catalogs are exported to `meta/` inside your data directory (`$CAMPFIRE_ROOT/meta/` or `~/campfire/meta/`). These are plain files you can load with pandas or astropy — no Python client needed.

In [ ]:
from astropy.table import Table

# Read the CSV catalogs directly (no client needed after sync)
# Default location: $CAMPFIRE_ROOT/meta/ or ~/campfire/meta/
objects = Table.read('~/campfire/meta/objects.csv')
spectra = Table.read('~/campfire/meta/spectra.csv')

print(f"Objects: {len(objects)} rows, columns: {objects.colnames}")
print(f"Spectra: {len(spectra)} rows, columns: {spectra.colnames}")

# Filter with astropy
high_z = objects[(objects['redshift'] > 5.0) & (objects['redshift_quality'] >= 2)]
print(f"\n{len(high_z)} objects at z > 5 with quality >= 2")

## Summary

| Method | Purpose |
|--------|---------|
| `cf.sync()` | Pull full catalog (metadata only, fast) |
| `cf.download()` | Bulk download FITS by obs/program/field/grating |
| `cf.query_objects()` | Filter and search the catalog (local SQLite) |
| `cf.iter_objects()` | Stream through large result sets |
| `cf.open_spectrum()` | Get spectral arrays (local first, API fallback) |
| `SpectrumData.from_fits()` | Open any FITS file directly |

See the [full documentation](https://campfire.hollisakins.com/docs/api) for the complete reference.